# Soldani - Second task - Benchmark

## 1. Setup del path di progetto

Individua automaticamente la directory radice del progetto cercando la cartella src/ nella gerarchia superiore, quindi la aggiunge a sys.path per consentire gli import assoluti.


In [54]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Import librerie e configurazione client LLM

Importa pandas, json, pgmpy e i moduli causali di FairMind (build_sfm, fit_discrete_bayesian_model, effetti). L'endpoint del server llama.cpp viene letto dalle variabili d'ambiente LLAMA_HOST/LLAMA_PORT (fallback localhost:8080), cosi' lo stesso notebook funziona sia in locale sia su THOR.


In [ ]:
import json
import os
import pandas as pd

from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect,
    natural_direct_effect, natural_indirect_effect,
)
from src.llm import LLM_CONFIGS

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"
print(f"LLM endpoint configurato: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")

## 3. Configurazione del benchmark

Definisce il dizionario CONFIG con il dataset Adult: attributo protetto (S2_gender, Female/Male), target (T_income, >50K), mediatore (hours-per-week), confounder (education).


In [56]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

## 4. FairMind — calcolo del ground truth

Costruisce lo Standard Fairness Model (SFM), fitta la Bayesian Network con stima BDeu e calcola TV, TE, DE, IE tramite inferenza causale formale. SE si ricava come `TV - TE` (Eq. 3, Plecko & Bareinboim 2024) — non da `spurious_effect()` (che calcola una quantità a un solo argomento, diversa dalla SE ufficiale a due argomenti). Restituisce anche la BN fittata: viene riusata in `build_llm_prompt()` per calcolare le tabelle date all'LLM, così FairMind e LLM partono dagli stessi identici numeri.


In [ ]:
import time

def run_fairmind(config: dict) -> tuple[dict, "DiscreteBayesianNetwork", int, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    # Binning di hours-per-week fatto qui, in-place, una sola volta: la BN
    # fittata su questi dati binnati e' la STESSA istanza che build_llm_prompt()
    # interroga per costruire le tabelle date all'LLM (Punto 4 del docente) —
    # cosi' LLM e FairMind partono dagli stessi identici numeri per ogni
    # cella (anche quelle piu' sparse), invece che uno da frequenze pandas
    # grezze e l'altro da un modello con smoothing BDeu.
    if "hours-per-week" in df.columns:
        df["hours-per-week"] = pd.cut(
            df["hours-per-week"],
            bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"],
            include_lowest=True,
        )

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    tv = total_variation(bn, target, config["protected"], x0, x1)
    te = total_effect(bn, target, config["protected"], x0, x1)
    effects = {
        "TV": tv,
        "TE": te,
        # SE = TV - TE (Eq. 3, Plecko & Bareinboim 2024) — NON spurious_effect()
        # da sola: quella calcola P(y|x)-P(y|do(x)) per un solo x, una quantita'
        # diversa dalla SE ufficiale a due argomenti usata nel resto del paper.
        "SE": tv - te,
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, bn, len(df), elapsed

ground_truth, bn, n_rows, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind — elapsed time: {fairmind_time:.4f}s")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

## 5. Costruzione del prompt per il LLM

Interroga la Bayesian Network già fittata (via `VariableElimination` di pgmpy) per costruire le 5 tabelle di probabilità condizionale (P(Y|X), P(Z), P(Y|X,Z), P(W|X,Z), P(Y|X,W,Z)) — non più frequenze empiriche pandas sul dataset grezzo. Assembla il prompt testuale con le formule di identificazione. Chiede solo TV, TE, DE, IE (non SE, ridondante: si ricava per sottrazione da TV e TE).


In [ ]:
from itertools import product

def _bn_states(bn, var: str) -> list:
    return bn.get_cpds(var).state_names[var]


def _bn_combos(bn, variables: list[str]) -> list[dict]:
    """Tutte le combinazioni congiunte di stati per una lista di variabili
    della BN, come lista di dict {variabile: stato}. Serve per enumerare le
    righe delle tabelle (una per combinazione) senza hardcodare gli stati."""
    if not variables:
        return [{}]
    state_lists = [_bn_states(bn, v) for v in variables]
    return [dict(zip(variables, combo)) for combo in product(*state_lists)]


def build_llm_prompt(config: dict, bn, n_rows: int) -> str:
    """Costruisce il prompt per l'LLM interrogando DIRETTAMENTE la Bayesian
    Network gia' fittata in run_fairmind() (stessa istanza, stesso smoothing
    BDeu), invece di ricalcolare le probabilita' con pandas sul dataset
    grezzo. Cosi' LLM e FairMind partono dagli stessi identici numeri per
    ogni cella delle tabelle, comprese quelle piu' sparse (Punto 4 del
    docente).
    """
    protected = config["protected"]
    target_var = config["target_col"]
    target_val = config["target_val"]
    confounders = config["confounders"]
    mediators = config["mediators"]
    x0, x1 = config["x0"], config["x1"]

    ve = VariableElimination(bn)

    # --- 1. P(Y=y | X) ---
    rows = []
    for x in [x0, x1]:
        f = ve.query(variables=[target_var], evidence={protected: x}, show_progress=False)
        p = float(f.get_value(**{target_var: target_val}))
        rows.append({protected: x, "P(Y=y|X)": round(p, 4)})
    p_y_given_x = pd.DataFrame(rows)

    # --- 2. P(Z) — distribuzione marginale dei confounders ---
    z_factor = ve.query(variables=confounders, joint=True, show_progress=False)
    rows = []
    for z_combo in _bn_combos(bn, confounders):
        p = float(z_factor.get_value(**z_combo))
        rows.append({**z_combo, "P(Z)": round(p, 4)})
    p_z = pd.DataFrame(rows)

    # --- 3. P(Y=y | X, Z) ---
    rows = []
    for x in [x0, x1]:
        for z_combo in _bn_combos(bn, confounders):
            f = ve.query(variables=[target_var], evidence={protected: x, **z_combo}, show_progress=False)
            p = float(f.get_value(**{target_var: target_val}))
            rows.append({protected: x, **z_combo, "P(Y=y|X,Z)": round(p, 4)})
    p_y_given_xz = pd.DataFrame(rows)

    # --- 4. P(W | X, Z) ---
    rows = []
    for x in [x0, x1]:
        for z_combo in _bn_combos(bn, confounders):
            f = ve.query(variables=mediators, evidence={protected: x, **z_combo}, joint=True, show_progress=False)
            for w_combo in _bn_combos(bn, mediators):
                p = float(f.get_value(**w_combo))
                rows.append({protected: x, **z_combo, **w_combo, "P(W|X,Z)": round(p, 4)})
    p_w_given_xz = pd.DataFrame(rows)

    # --- 5. P(Y=y | X, W, Z) ---
    rows = []
    for x in [x0, x1]:
        for z_combo in _bn_combos(bn, confounders):
            for w_combo in _bn_combos(bn, mediators):
                f = ve.query(variables=[target_var], evidence={protected: x, **z_combo, **w_combo}, show_progress=False)
                p = float(f.get_value(**{target_var: target_val}))
                rows.append({protected: x, **w_combo, **z_combo, "P(Y=y|X,W,Z)": round(p, 4)})
    p_y_given_xwz = pd.DataFrame(rows)

    def to_compact_csv(d: pd.DataFrame) -> str:
        return d.to_csv(index=False)

    n_z = len(_bn_combos(bn, confounders))
    z_example = _bn_combos(bn, confounders)[0][confounders[0]] if confounders else ""

    return f"""You are a causal fairness expert. Compute four causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

You are given PRE-AGGREGATED CONDITIONAL PROBABILITY TABLES computed from a fitted
Bayesian Network (n={n_rows} training rows, BDeu-smoothed CPDs). Use these tables
directly — do not assume access to raw data.
Note: "hours-per-week" has been discretized into bins: <=20, 21-35, 36-45, 46-60, >60.

VARIABLE ROLES:
- X (protected): "{protected}", x0="{x0}", x1="{x1}"
- Y (target):    "{target_var}", target state="{target_val}"
- W (mediators): {mediators}
- Z (confounders): {confounders}

TABLE 1 — P(Y=y | X):
{to_compact_csv(p_y_given_x)}

TABLE 2 — P(Z):
{to_compact_csv(p_z)}

TABLE 3 — P(Y=y | X, Z):
{to_compact_csv(p_y_given_xz)}

TABLE 4 — P(W | X, Z):
{to_compact_csv(p_w_given_xz)}

TABLE 5 — P(Y=y | X, W, Z):
{to_compact_csv(p_y_given_xwz)}

IDENTIFICATION FORMULAE (use these exactly, aggregating over TABLE rows as needed):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)                    [from TABLE 1]
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)          [from TABLE 3, TABLE 2]
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)   [from TABLE 5, TABLE 4, TABLE 2]
- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)       [from TABLE 5, TABLE 4, TABLE 2]

Note: the Spurious Effect (SE) is NOT requested here — it is fully determined
by SE = TV - TE (Plecko & Bareinboim, 2024), so it is derived afterwards
from your TV and TE values rather than computed independently.

INSTRUCTIONS:
For DE and IE, the sums run over EVERY combination of z (each row of TABLE 2,
{n_z} values) and w (each bin of hours-per-week) — do not skip or approximate
any (z,w) term.

You MUST actually compute every (z,w) term — do not guess or shortcut the
result. To keep the response short, show your work GROUPED BY z, one COMPACT
plain-text line per z (no LaTeX, no "\\[", no "\\cdot", no markdown — use "*"
for multiplication and "+" between terms), listing the (z,w) products for that
z and their sum, like this exact style:
{z_example}: (a-b)*c*d + (e-f)*g*d + ... = <subtotal>

Produce exactly {n_z} such lines for DE (one per z), then exactly {n_z} more
for IE (one per z). Do not add any other commentary between the lines.

After the two lists of {n_z} lines each (DE list, then IE list), give the final answer.

End your response with a line "FINAL_JSON:" followed by ONLY the JSON object below,
with no markdown formatting:
{{
  "TV": <float>,
  "TE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""

prompt = build_llm_prompt(CONFIG, bn, n_rows)
print(prompt[:2000], "\n[...]")
print(f"\nTotal prompt length (chars): {len(prompt)}")

## 6. Chiamata al LLM (Qwen2.5-7B) e parsing della risposta

Invia il prompt con le tabelle pre-aggregate al modello Qwen2.5-7B via llama.cpp, raccoglie le metriche di tempo e token, estrae il JSON (TV, TE, DE, IE) dalla risposta tramite regex e lo parsifica. SE viene poi calcolata come `TV - TE` sui valori restituiti dall'LLM.


In [ ]:
from src.llm import call_llm

# max_tokens=16384: con "education" come confounder (16 livelli) anche il
# formato compatto (una riga per z, testo semplice invece di LaTeX) richiede
# comunque ~16 righe per DE + 16 per IE con dentro il lavoro reale — 8192 non
# bastava con la formattazione LaTeX di prima, meglio avere margine ampio
# piuttosto che un quinto tentativo fallito.
llm_effects, llm_usage, llm_time = call_llm(prompt, max_tokens=16384)

# SE non viene chiesta all'LLM (v. nota nel prompt): si ricava qui con la
# stessa identita' usata per il ground truth (SE = TV - TE), cosi' il
# confronto sulla SE riflette solo gli errori di TV/TE dell'LLM, non un
# calcolo extra e ridondante.
llm_effects["SE"] = llm_effects["TV"] - llm_effects["TE"]

print(f"LLM — time: {llm_time:.4f}s")
print(f"Token: input={llm_usage['input_tokens']}, "
      f"output={llm_usage['output_tokens']}, "
      f"total={llm_usage['total_tokens']}")
print(json.dumps(llm_effects, indent=2))

## 7. Confronto FairMind vs LLM — tabella discrepancies

Calcola l'errore assoluto e relativo percentuale tra il ground truth (FairMind) e la predizione del LLM per ognuno dei 5 effetti causali, producendo una tabella riassuntiva.


In [60]:
def compute_discrepancies(ground_truth: dict, llm_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt  = ground_truth.get(effect, float("nan"))
        llm_val = float(llm_effects.get(effect, float("nan")))
        abs_err = abs(gt - llm_val)
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect":      effect,
        "fairmind":    round(gt,  6),
        "llm":         round(llm_val, 6),
            "abs_error":   round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)

discrepancies = compute_discrepancies(ground_truth, llm_effects)
print(discrepancies.to_string(index=False))

effect  fairmind       gpt  abs_error  rel_error_%
    TV  0.194470  0.194500   0.000030         0.02
    TE  0.183161  0.231167   0.048005        26.21
    SE -0.007296 -0.036667   0.029371       402.56
    DE  0.137049  0.083167   0.053883        39.32
    IE -0.046112 -0.047000   0.000888         1.93


## 8. Salvataggio dei risultati su file JSON

Salva l'intero risultato del benchmark in un file JSON dentro benchmark_results/, includendo configurazione, effetti FairMind, effetti LLM, discrepancies, metriche token e timing.


In [61]:
def save_results(config, ground_truth, llm_effects, discrepancies, usage, fairmind_time, llm_time):
    import os, datetime
    os.makedirs("benchmark_results", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/{config['dataset_name']}_{ts}.json"

    out = {
        "dataset":       config["dataset_name"],
        "config":        {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind":      ground_truth,
        "llm":           llm_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage":   usage,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "llm_seconds":      round(llm_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved: {fname}")

save_results(CONFIG, ground_truth, llm_effects, discrepancies, llm_usage, fairmind_time, llm_time)

Saved: benchmark_results/adult_20260712_161949.json
